# VALORANT 前日試合ふり返り
ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選んでから、上から2つ実行するだけです。
設定は `config/default.yaml`、上書きは Drive の `valorant-replay/config.yaml` に書きます。

In [ ]:
!git -C /content/valorant-replay pull -q 2>/dev/null || git clone -q https://github.com/AK47bbddk/valorant-replay /content/valorant-replay
!pip -q install -e '/content/valorant-replay[video]'
!test -x ~/.deno/bin/deno || (curl -fsSL https://deno.land/install.sh | sh -s -- -y > /dev/null 2>&1)  # YouTube の解読に使う JS ランタイム
%cd /content/valorant-replay

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
def secret(k, required=False):
    try:
        v = (userdata.get(k) or '').strip()
    except Exception as e:   # 未登録・ノートブックからのアクセスがオフ・名前違い など
        if required:
            print(f'⚠ シークレット {k} を読めません: {type(e).__name__} {e}')
        return ''
    if required:
        print(f'✓ {k} を読み込みました（{len(v)} 文字, 先頭 {v[:4]}…）')
    return v
os.environ['NOTION_TOKEN'] = secret('NOTION_TOKEN', required=True)
os.environ['YT_COOKIES'] = secret('YT_COOKIES')  # 任意。Drive の valorant-replay/cookies.txt でもよい
os.environ['VREPLAY_CONFIG'] = '/content/drive/MyDrive/valorant-replay/config.yaml'  # 無ければ既定値で動く
!vreplay run   # 例: --date 2026-10-21 / --match <VLRのURL> / --no-video / --force

### ROI の確認（HUD の位置がずれているとき）
下のセルの URL を好きな試合に変えて実行すると、枠を描いた画像がその場に表示され、ダウンロードもされます。

※ ランタイムを切り替える・切断する前に最後のセルを実行してください（Drive への書き込みが反映されずに消えることがあります）。

In [ ]:
!vreplay calibrate --match https://www.vlr.gg/754737/loud-vs-team-vitality-valorant-champions-2026-gf --map 1 --at 600
import os
from IPython.display import Image, display
from google.colab import files
d = '/content/drive/MyDrive/valorant-replay/calibrate/'
for n in ('rois.jpg', 'panels.jpg'):
    if os.path.exists(d + n):
        display(Image(d + n))
        files.download(d + n)

In [ ]:
drive.flush_and_unmount()  # Drive への書き込みを確定（終わる前・ランタイムを変える前に）